In [1]:
import hashlib
import os
import platform
import sys
import uuid

# 修改为自己的学号
STUDENT_ID = 20244080304
STUDENT_NAME = "李晓月"

CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV")
print(f"[Conda 环境] {CONDA_ENV}")

MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]

RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[姓名]       {STUDENT_NAME}")
print(f"[学号]       {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID]   {RUN_ID}")
print(f"[Python]    {sys.version.split()[0]}")

[Conda 环境] exp01_env
[姓名]       李晓月
[学号]       20244080304
[机器指纹] 5fc15abbc5865ff7
[RUN_ID]   20244080304-5fc15abbc5865ff7
[Python]    3.10.22


## 实验03：正则表达式与网络结构化数据

**学号 / 姓名**：20244080304 / 李晓月

本实验分三部分：正则表达式的基础匹配、分组 / 回溯引用 / 替换、以及用 BeautifulSoup 解析静态 HTML。
所有输出均由代码单元格直接产生，说明性文字放在 Markdown 单元格中。

---

## 实验1：正则表达式的基础匹配

先导入 `re`。以下每个小任务单独用一行输出，便于对照「模式 → 匹配结果」。

In [2]:
import re

text = "cat on mat; later: cats and dogs"
print("待匹配文本：", repr(text))
print(r"\d+      ->", re.findall(r"\d+", text))
print(r"\s+      ->", re.findall(r"\s+", text))
print(r"\bcat\b  ->", re.findall(r"\bcat\b", text))

待匹配文本： 'cat on mat; later: cats and dogs'
\d+      -> []
\s+      -> [' ', ' ', ' ', ' ', ' ', ' ']
\bcat\b  -> ['cat']


In [3]:
text2 = "cat dog bird CAT"
print("待匹配文本：", repr(text2))
print(r"cat|dog        ->", re.findall(r"cat|dog", text2))
print(r"cat|dog (忽略大小写) ->", re.findall(r"cat|dog", text2, flags=re.IGNORECASE))

s = "aaaa"
print()
print("展示交替（|）的“最左优先 + 按书写顺序”规则：")
print(r'a|aa|aaa ->', re.findall(r"a|aa|aaa", s))
print(r'aaa|aa|a ->', re.findall(r"aaa|aa|a", s))

待匹配文本： 'cat dog bird CAT'
cat|dog        -> ['cat', 'dog']
cat|dog (忽略大小写) -> ['cat', 'dog', 'CAT']

展示交替（|）的“最左优先 + 按书写顺序”规则：
a|aa|aaa -> ['a', 'a', 'a', 'a']
aaa|aa|a -> ['aaa', 'a']


**关于交替顺序的差异**

`re.findall` 从左到右扫描，在每个起始位置上按**书写顺序**依次尝试各个分支，一旦某个分支匹配成功就接受并继续扫描。

- 模式 `a|aa|aaa`：在位置 0 先试 `a` 就成功，于是逐字符吃掉，得到 4 个 `'a'`；
- 模式 `aaa|aa|a`：在位置 0 先试 `aaa` 成功，剩 1 个 `a` 再由 `a` 吃掉，得到 `['aaa', 'a']`。

两者匹配到的**字符总数相同**（都是 4 个 `a`），但切分方式不同。这说明正则引擎不做「最长匹配」的全局优化，结果依赖分支顺序。

In [4]:
t = "ct cat caat caaat"
print("待匹配文本：", repr(t))
for pat in [r"ca*t", r"ca+t", r"ca?t", r"ca{2}t", r"ca{1,2}t"]:
    print(f"{pat:10s} -> {re.findall(pat, t)}")

待匹配文本： 'ct cat caat caaat'
ca*t       -> ['ct', 'cat', 'caat', 'caaat']
ca+t       -> ['cat', 'caat', 'caaat']
ca?t       -> ['ct', 'cat']
ca{2}t     -> ['caat']
ca{1,2}t   -> ['cat', 'caat']


**各量词的含义与匹配结果**

| 模式 | 含义 | 在 `ct cat caat caaat` 上的匹配 |
|---|---|---|
| `ca*t` | `a` 出现 **0 次或多次** | `ct`、`cat`、`caat`、`caaat` 全部命中 |
| `ca+t` | `a` 出现 **1 次或多次** | `cat`、`caat`、`caaat`，`ct` 不匹配 |
| `ca?t` | `a` 出现 **0 次或 1 次** | `ct`、`cat`；`caat`（两个 `a`）和 `caaat`（三个 `a`）都不匹配 |
| `ca{2}t` | `a` **恰好 2 次** | 只有 `caat` |
| `ca{1,2}t` | `a` **1 到 2 次** | `cat`、`caat`；`caaat` 中 `caa` 之后紧跟的是 `a` 而不是 `t`，故整体不匹配 |

量词默认是**贪婪**的：`ca+t` 在 `caaat` 上会把 3 个 `a` 全部吃掉（匹配 `caaat`），而不是只吃 1 个 `a` 就停下。

---

## 实验2：分组、回溯引用与替换

In [5]:
mail_text = "联系：alice@example.com；备选：bob@test.org"
# 第 1 组：用户名；第 2 组：域名（含点号分隔的各级域名）
mail_re = re.compile(r"([\w.+-]+)@([\w-]+(?:\.[\w-]+)+)")

print("待匹配文本：", mail_text)
print()
for m in mail_re.finditer(mail_text):
    print(f"整体匹配：{m.group(0):<20} 用户名={m.group(1):<8} 域名={m.group(2)}")

print()
print("findall 直接返回分组元组：", mail_re.findall(mail_text))
users, domains = zip(*mail_re.findall(mail_text))
print("用户名列表：", list(users))
print("域名列表：  ", list(domains))

待匹配文本： 联系：alice@example.com；备选：bob@test.org

整体匹配：alice@example.com    用户名=alice    域名=example.com
整体匹配：bob@test.org         用户名=bob      域名=test.org

findall 直接返回分组元组： [('alice', 'example.com'), ('bob', 'test.org')]
用户名列表： ['alice', 'bob']
域名列表：   ['example.com', 'test.org']


In [6]:
words = "cat cat, dog cat, bird bird"
# \1 回溯引用第 1 个分组，要求前后两个单词完全相同
dup_re = re.compile(r"\b(\w+)\s+\1\b")

print("待匹配文本：", words)
print("重复单词：", [m.group(0) for m in dup_re.finditer(words)])
print("捕获到的词：", dup_re.findall(words))
print("去重后：   ", dup_re.sub(r"\1", words))

待匹配文本： cat cat, dog cat, bird bird
重复单词： ['cat cat', 'bird bird']
捕获到的词： ['cat', 'bird']
去重后：    cat, dog cat, bird


In [7]:
dates = "2024-05-01\n2024/05/02\n2024.05.03"
# 三种分隔符都接受：[-/.]
date_re = re.compile(r"(\d{4})([-/.])(\d{2})\2(\d{2})")
new_dates = date_re.sub(r"\3-\4-\1", dates)

print("转换前：")
print(dates)
print()
print("转换后（月-日-年）：")
print(new_dates)
print()
print("行数：", len(dates.splitlines()), "->", len(new_dates.splitlines()))

转换前：
2024-05-01
2024/05/02
2024.05.03

转换后（月-日-年）：
05-01-2024
05-02-2024
05-03-2024

行数： 3 -> 3


**说明**

- 替换表达式 `\3-\4-\1` 按「第 3 组（月）- 第 4 组（日）- 第 1 组（年）」重排；
- 分组 2 是分隔符，这里用 `\2` 回溯引用，保证 `2024-05-01` 这类**同一种分隔符**才匹配，同时也能接受 `/` 和 `.`；
- `re.sub` 只替换匹配到的片段，`\n` 不在匹配范围内，因此**行数保持 3 行**。

In [8]:
phones = "联系我：123-456-7890 或 987.654.3210"
# 第 1、2 段用 [-.] 分隔，末段 4 位纯数字
phone_re = re.compile(r"(\d{3})[-.](\d{3})[-.](\d{4})")
out = phone_re.sub(r"(\1) \2-\3", phones)

print("转换前：", phones)
print("转换后：", out)
print()
print("提取到的号码：", ["".join(g) for g in phone_re.findall(phones)])

转换前： 联系我：123-456-7890 或 987.654.3210
转换后： 联系我：(123) 456-7890 或 (987) 654-3210

提取到的号码： ['1234567890', '9876543210']


---

## 实验3：使用 BeautifulSoup 解析 HTML

使用题目给定的字符串，不依赖网页下载。

In [9]:
from bs4 import BeautifulSoup

html_doc = """
<html><head><title>实验课程</title></head><body>
<h1>数据分析</h1>
<p class="course">第六讲：正则表达式</p>
<p class="course">第七讲：结构化数据</p>
<a href="https://example.org/lecture06">讲义六</a>
<a href="https://example.org/lecture07">讲义七</a>
</body></html>
"""

soup = BeautifulSoup(html_doc, "html.parser")

print("页面标题文本：", soup.title.get_text())
print("h1 标签名：   ", soup.h1.name)
print("h1 文本：     ", soup.h1.get_text())

页面标题文本： 实验课程
h1 标签名：    h1
h1 文本：      数据分析


In [10]:
# 1) 所有 class="course" 的段落
print('class="course" 的段落：')
for i, p in enumerate(soup.find_all("p", class_="course"), 1):
    print(f"  {i}. {p.get_text()}")

# 2) 所有链接：文字 + href
print()
print("所有链接：")
for i, a in enumerate(soup.find_all("a"), 1):
    print(f"  {i}. 文字={a.get_text()!r}  href={a.get('href')!r}")

links = soup.find_all("a")
print()
print("链接总数：", len(links))
print("href 列表：", [a["href"] for a in links])

class="course" 的段落：
  1. 第六讲：正则表达式
  2. 第七讲：结构化数据

所有链接：
  1. 文字='讲义六'  href='https://example.org/lecture06'
  2. 文字='讲义七'  href='https://example.org/lecture07'

链接总数： 2
href 列表： ['https://example.org/lecture06', 'https://example.org/lecture07']


In [11]:
whole = soup.get_text()
para = soup.find("p", class_="course").get_text()
link = soup.find("a").get_text()

print("整个文档的可见文本：")
print(repr(whole))
print()
print("单个段落：", repr(para))
print("单个链接：", repr(link))
print()
print("--- 对比 ---")
print(f"文档文本长度 = {len(whole)}，段落长度 = {len(para)}，链接长度 = {len(link)}")
print("文档文本去掉首尾空白后的行：")
for line in whole.strip().splitlines():
    print("   ", repr(line))

整个文档的可见文本：
'\n实验课程\n数据分析\n第六讲：正则表达式\n第七讲：结构化数据\n讲义六\n讲义七\n\n'

单个段落： '第六讲：正则表达式'
单个链接： '讲义六'

--- 对比 ---
文档文本长度 = 40，段落长度 = 9，链接长度 = 3
文档文本去掉首尾空白后的行：
    '实验课程'
    '数据分析'
    '第六讲：正则表达式'
    '第七讲：结构化数据'
    '讲义六'
    '讲义七'


**解析结果说明**

1. `soup.title` / `soup.h1` 直接按标签名取到**第一个**同名标签；`get_text()` 返回其内部纯文本，标签本身（如 `h1`）不进入结果，所以 `soup.h1.name == 'h1'` 而文本是 `数据分析`。
2. `find_all("p", class_="course")` 用属性过滤，得到两个段落；`find_all("a")` 得到两个 `<a>`，其 `href` 通过字典式取值 `a["href"]`（或 `a.get("href")`）获得。
3. `get_text()` 只返回**可见文本**，会丢掉所有标签，并把相邻元素之间的文本拼接起来。因此整篇文档的 `get_text()` 是标题、h1、两段正文、两个链接文字的连写（含源码中元素之间的空白），长度远大于单个段落或单个链接；单元素调用只返回该元素内部的文字。